<a href="https://colab.research.google.com/github/Diego-1099/Colabfiles/blob/main/Dashboard_en_Gooble_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# ==============================================================================
# 1.                  LIBRERÍAS Y CONFIGURACIÓN INICIAL
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR

from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings("ignore")

out = widgets.Output()

print("☑️ Entorno preparado y librerías cargadas.")

# ==============================================================================
# 2 y 3.                 CARGA Y CONFIGURACIÓN DINÁMICA
# ==============================================================================

uploader = widgets.FileUpload(
    accept = ".csv",
    multiple = False,
    description = "Subir archivo CSV",
    button_style = "info"
)

# Widgets con estilo ajustado
dropdown_fecha = widgets.Dropdown(description = "Col. Fecha:")
dropdown_datos = widgets.Dropdown(description = "Col. Datos:")
selector_features = widgets.SelectMultiple(
    options = ['Mes', 'Año', 'Trimestre', 'Día_semana', 'Lag_1'],
    value = ['Mes', 'Año'],
    description = 'Features:',
)

# Contenedor para organizar los widgets verticalmente
ui_configuracion = widgets.VBox([
    widgets.HTML("<b>--- PASO 2: CONFIGURACIÓN DE COLUMNAS Y FEATURES ---</b>"),
    dropdown_fecha,
    dropdown_datos,
    selector_features,
    widgets.HTML("<i>➡️ Configura los campos y pasa a la siguiente celda.</i>")
])

def procesar_flujo_completo(change):
    global df_dashboard
    with out:
        clear_output()

        # 1. Leer el archivo
        input_file = list(uploader.value.values())[0]
        content = input_file['content']
        df_dashboard = pd.read_csv(io.BytesIO(content))

        print(f"✔️ Archivo '{input_file['metadata']['name']}' cargado con éxito.")
        display(df_dashboard.head(5))

        # 2. Actualizar los menús
        columnas = df_dashboard.columns.tolist()
        dropdown_fecha.options = columnas
        dropdown_datos.options = columnas

        if 'Month' in columnas: dropdown_fecha.value = 'Month'
        if 'Passengers' in columnas: dropdown_datos.value = 'Passengers'

        # 3. Mostrar la UI organizada
        display(ui_configuracion)

uploader.observe(procesar_flujo_completo, names = 'value')

display(uploader, out)

# ==============================================================================
# 4 y 5. HORIZONTE Y MÉTRICAS (Instrucciones 4 y 5)
# ==============================================================================

# Instrucción 4: Slider para el horizonte
slider_horizonte = widgets.IntSlider(
    value = 12,
    min = 1,
    max = 48,
    step = 1,
    description = 'Horizonte:',
    continuous_update = False,
    style={'description_width': 'initial'}
)

# Instrucción 5: Selección de métricas
selector_metricas = widgets.SelectMultiple(
    options=['MSE', 'MAE', 'RMSE', 'MAPE', 'sMAPE'],
    value=['MAE', 'RMSE', 'MAPE'],
    description='Métricas:',
    style={'description_width': 'initial'}
)


ui_evaluacion = widgets.VBox([
    widgets.HTML("<br><b>--- PASO 3: DEFINIR PARÁMETROS DE EVALUACIÓN ---</b>"),
    slider_horizonte,
    selector_metricas,
    widgets.HTML("<i>➡️ Define el horizonte de pronóstico y las métricas para evaluar los modelos.</i>")
])

display(ui_evaluacion)

# ==============================================================================
# 6 y 7. SELECCIÓN DE MODELOS Y EJECUCIÓN (Instrucciones 6 y 7)
# ==============================================================================

# Instrucción 6: Selección de al menos 3 modelos de ML
selector_modelos = widgets.SelectMultiple(
    options=['Random Forest', 'Gradient Boosting', 'Linear Regression', 'SVR'],
    value=['Random Forest', 'Gradient Boosting', 'Linear Regression'],
    description='Modelos:',
    style={'description_width': 'initial'}
)

# Botón para ejecutar el análisis
boton_ejecutar = widgets.Button(
    description="Generar Dashboard",
    button_style='success', # Color verde
    icon='check'
)

# Contenedor visual consistente
ui_modelos = widgets.VBox([
    widgets.HTML("<br><b>--- PASO 4: SELECCIÓN DE MODELOS Y VISUALIZACIÓN ---</b>"),
    selector_modelos,
    widgets.HTML("<br>"),
    boton_ejecutar,
    widgets.HTML("<i><br>➡️ Selecciona los modelos que deseas visualizar y presiona el botón.</i>")
])

# Función para calcular las métricas seleccionadas
def calcular_metricas_seleccionadas(y_true, y_pred, seleccionadas):
    res = {}
    if 'MAE' in seleccionadas: res['MAE'] = mean_absolute_error(y_true, y_pred)
    if 'MSE' in seleccionadas: res['MSE'] = mean_squared_error(y_true, y_pred)
    if 'RMSE' in seleccionadas: res['RMSE'] = np.sqrt(mean_squared_error(y_true, y_pred))
    if 'MAPE' in seleccionadas: res['MAPE'] = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    if 'sMAPE' in seleccionadas: res['sMAPE'] = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))
    return res

# Función que se ejecuta al dar click
def ejecutar_dashboard(b):
    with out:
        print("\n⏳ Procesando modelos y generando gráficas...")

        # Preparación de datos
        data = df_dashboard.copy()
        data[dropdown_fecha.value] = pd.to_datetime(data[dropdown_fecha.value])
        data = data.sort_values(dropdown_fecha.value).set_index(dropdown_fecha.value)

        # Target
        target = dropdown_datos.value

        # Ingeniería de Features (Punto 3)
        if 'Mes' in selector_features.value: data['Mes'] = data.index.month
        if 'Año' in selector_features.value: data['Año'] = data.index.year
        if 'Trimestre' in selector_features.value: data['Trimestre'] = data.index.quarter
        if 'Día_semana' in selector_features.value: data['Día_semana'] = data.index.dayofweek
        if 'Lag_1' in selector_features.value: data['Lag_1'] = data[target].shift(1)

        data = data.dropna()
        X = data.drop(columns=[target])
        y = data[target]

        # Split Train/Test basado en el Horizonte (Punto 4)
        h = slider_horizonte.value
        X_train, X_test = X[:-h], X[-h:]
        y_train, y_test = y[:-h], y[-h:]

        # Graficación (Punto 7)
        plt.figure(figsize=(12, 6))
        plt.plot(y, label='Datos Reales', color='black', linewidth=1.5)

        resultados_finales = []

        # Entrenamiento de modelos (Punto 6)
        for m_name in selector_modelos.value:
            if m_name == 'Random Forest': model = RandomForestRegressor(n_estimators=100).fit(X_train, y_train)
            elif m_name == 'Gradient Boosting': model = GradientBoostingRegressor().fit(X_train, y_train)
            elif m_name == 'Linear Regression': model = LinearRegression().fit(X_train, y_train)
            elif m_name == 'SVR': model = SVR().fit(X_train, y_train)

            preds = model.predict(X_test)
            plt.plot(y_test.index, preds, label=f'Pronóstico {m_name}', linestyle='--')

            # Métricas (Punto 5)
            m_vals = calcular_metricas_seleccionadas(y_test, preds, selector_metricas.value)
            m_vals['Modelo'] = m_name
            resultados_finales.append(m_vals)

        plt.title(f"Dashboard Interactiva: Predicción de {target}")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

        # Mostrar tabla de resultados
        df_res = pd.DataFrame(resultados_finales).set_index('Modelo')
        display(df_res)

boton_ejecutar.on_click(ejecutar_dashboard)
display(ui_modelos)

☑️ Entorno preparado y librerías cargadas.


FileUpload(value={}, accept='.csv', button_style='info', description='Subir archivo CSV')

Output()